# COMP8410 A1 - Week 8: DOM-text agent vs screenshot agent (controlled pair) on MiniWoB++ - Colab
Agents: **rule-based baseline** (CPU), **Qwen2.5-VL-7B on DOM text**, **UI-TARS-1.5-7B on screenshots** (a GUI-trained Qwen2.5-VL derivative).
8 tasks x 3 seeds clean, plus a prompt-injection variant on 4 tasks. **Budget for this pair: about 6 units.** Phase 1 needs no GPU; phase 2 needs a 24 GB GPU (L4) or an A100 (the 7B models are about 17 GB in bf16).

| Section | What | Runtime |
|---|---|---|
| 1 | Setup | any |
| 2 | Browser + MiniWoB install, CPU tests, rule-agent smoke (this proves the browser, screenshots and page-clock fix work) | **CPU or T4** |
| 3 | Rule baseline, full | CPU or T4 |
| 4 | Model smoke tests + budget gate | L4 / A100 |
| 5 | Full runs (clean and injection) + report | L4 / A100 |
| 6 | Cost log, disconnect | end |

Tick *Omit code cell output when saving*. If a cell fails, stop and read it. Do not use Run all. **Disconnect when finished (6.2).**

## 1. Setup

In [ ]:
# [1.1] SETUP, fresh runtime only: mount Drive, clone the repo, link results/ and outputs/ to Drive
%cd /content
import os
os.environ['HF_HOME'] = '/content/hf'
from google.colab import drive # type: ignore
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/comp8410_a1'
for d in ['results', 'outputs', 'logs']:
    os.makedirs(f'{BASE}/{d}', exist_ok=True)
!rm -rf /content/a1 && git clone https://github.com/yash1998-MQ-AI/COMP8410-A1-48576158.git /content/a1
%cd "/content/a1/48576158 Evaluation"
!rm -rf results outputs && ln -s {BASE}/results results && ln -s {BASE}/outputs outputs
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# [1.2] RESTORE after a kernel restart (use INSTEAD of 1.1)
import os, json
os.environ['HF_HOME'] = '/content/hf'
BASE = '/content/drive/MyDrive/comp8410_a1'
%cd "/content/a1/48576158 Evaluation"
from google.colab import drive
drive.mount('/content/drive')
!git -C /content/a1 pull -q
b = json.load(open('/content/w8_browser.json'))
os.environ['MINIWOB_CHROME_BINARY'] = b['chrome']; os.environ['MINIWOB_CHROMEDRIVER'] = b['driver']

## 2. Browser + MiniWoB (CPU is enough)

In [ ]:
# [2.1] INSTALL miniwob, selenium, Playwright's Chromium (brings the system libraries) and a matching chromedriver (about 3-5 min)
%cd "/content/a1/48576158 Evaluation"
!python pairs/w8_agents/install_w8.py 2>&1 | tee {BASE}/logs/w8_install.log | tail -15

In [ ]:
# [2.2] BROWSER CHECK + CPU TESTS: opens a MiniWoB task, prints the observation keys (we need screenshot, dom_elements) and the page size
import os, json
b = json.load(open('/content/w8_browser.json'))
os.environ['MINIWOB_CHROME_BINARY'] = b['chrome']; os.environ['MINIWOB_CHROMEDRIVER'] = b['driver']
%cd "/content/a1/48576158 Evaluation"
!python -m pytest -q tests/test_w8.py
import gymnasium as gym, miniwob
env = gym.make('miniwob/click-button-v1', render_mode=None)
obs, _ = env.reset(seed=0)
print('keys:', list(obs.keys()), '| screenshot', obs['screenshot'].shape, '| utterance:', obs['utterance'])
print('dom:', [(int(e['ref']), e['tag'], e['text']) for e in obs['dom_elements'] if e['text']][:8])
env.close()

In [ ]:
# [2.3] SMOKE THE RULE AGENT (2 tasks x 1 seed). If it solves click-button the whole pipeline works (browser, actions, binary reward, page-clock fix).
%cd "/content/a1/48576158 Evaluation"
!python pairs/w8_agents/run_w8.py rule --smoke 2>&1 | grep -v Warning | tail -12

## 3. Rule baseline, full (CPU)

In [ ]:
# [3.1] RULE BASELINE: 8 tasks x 3 seeds clean, then the injection variant (4 tasks x 3 seeds)
%cd "/content/a1/48576158 Evaluation"
!python pairs/w8_agents/run_w8.py rule 2>&1 | tee {BASE}/logs/w8_rule_clean.log | grep -v Warning | tail -30
!python pairs/w8_agents/run_w8.py rule --inject 2>&1 | tee {BASE}/logs/w8_rule_inject.log | grep -v Warning | tail -15

## 4. Model smoke tests and the budget gate (GPU runtime: L4 or A100)

In [ ]:
# [4.1] SMOKE qwen_dom (DOM text): 2 tasks x 1 seed; downloads the model (about 16 GB) on first use
%cd "/content/a1/48576158 Evaluation"
!python pairs/w8_agents/run_w8.py qwen_dom --smoke 2>&1 | tee {BASE}/logs/w8_smoke_qwen.log | grep -v Warning | tail -20

In [ ]:
# [4.2] SMOKE uitars_vis (screenshots): 2 tasks x 1 seed
%cd "/content/a1/48576158 Evaluation"
!python pairs/w8_agents/run_w8.py uitars_vis --smoke 2>&1 | tee {BASE}/logs/w8_smoke_uitars.log | grep -v Warning | tail -20

In [ ]:
# [4.3] READ THE SMOKE TRAJECTORIES: what each agent saw and did (raw model text). Check the UI-TARS clicks land on the right element.
import json, glob
for f in sorted(glob.glob('outputs/w8_smoke/*/clean/*.json')):
    r = json.load(open(f))
    print('==', f.split('w8_smoke/')[1], 'success', r['success'], 'steps', r['n_steps'], 'invalid', r['invalid'], f"{r['latency_per_step_s']:.1f}s/step")
    for s in r['steps']:
        print('  ', s['step'], s['act'], '|', s['raw'][:160].replace('\n', ' / '))

In [ ]:
# [4.4] BUDGET GATE: projects the full model runs (clean 24 episodes + injection 12 episodes per agent) from the smoke timings
import json, glob
UNITS_PER_HR = 5.0       # L4 about 5? A100 11.77: READ THE RESOURCES PANEL AND EDIT
PAIR_BUDGET = 6.0
lat = []
for f in glob.glob('outputs/w8_smoke/*/clean/*.json'):
    r = json.load(open(f))
    if r['steps']:
        lat.append(r['latency_s'])
mean_ep = sum(lat) / max(len(lat), 1)
est_s = 36 * 2 * mean_ep + 2 * 240          # 2 agents x 36 episodes + loads
print(f'mean episode {mean_ep:.1f} s -> projected {est_s/60:.0f} min = {est_s/3600*UNITS_PER_HR:.1f} units (budget {PAIR_BUDGET})')
print('GATE', 'PASSED' if est_s / 3600 * UNITS_PER_HR <= PAIR_BUDGET else 'FAILED: cut seeds to 2 or tasks before 5.x')

## 5. Full runs and report

In [ ]:
# [5.1] FULL qwen_dom: clean (8 tasks x 3 seeds) then injection (4 tasks x 3 seeds). Resumable.
%cd "/content/a1/48576158 Evaluation"
!python pairs/w8_agents/run_w8.py qwen_dom 2>&1 | tee {BASE}/logs/w8_qwen_clean.log | grep -v Warning | tail -30
!python pairs/w8_agents/run_w8.py qwen_dom --inject 2>&1 | tee {BASE}/logs/w8_qwen_inject.log | grep -v Warning | tail -15

In [ ]:
# [5.2] FULL uitars_vis: clean then injection. Resumable.
%cd "/content/a1/48576158 Evaluation"
!python pairs/w8_agents/run_w8.py uitars_vis 2>&1 | tee {BASE}/logs/w8_uitars_clean.log | grep -v Warning | tail -30
!python pairs/w8_agents/run_w8.py uitars_vis --inject 2>&1 | tee {BASE}/logs/w8_uitars_inject.log | grep -v Warning | tail -15

In [ ]:
# [5.3] TABLES AND PLOTS (copies report assets and pip freeze to Drive)
%cd "/content/a1/48576158 Evaluation"
!python pairs/w8_agents/report_w8.py --show 2>&1 | tee {BASE}/logs/w8_report.log | grep -v Warning
!mkdir -p {BASE}/report_assets && cp -r report_assets/w8 {BASE}/report_assets/
!pip freeze > {BASE}/results/w8_ENV.txt

## 6. Cost, shut down

In [ ]:
# [6.1] COST LOG: fill in HOURS and UNITS_PER_HR (Resources panel), run once per session
%cd "/content/a1/48576158 Evaluation"
HOURS = None
UNITS_PER_HR = None
if HOURS and UNITS_PER_HR:
    from a1eval import log_cost
    log_cost('cost_log.csv', '2026-11-03', 'Colab Pro', 'L4/A100', HOURS, UNITS_PER_HR * 0.1513, note='W8 install + agents on MiniWoB++')
    !tail -3 cost_log.csv

In [ ]:
# [6.2] FREE THE GPU: set AUTO_DISCONNECT = True once everything is copied to Drive
AUTO_DISCONNECT = False
if AUTO_DISCONNECT:
    from google.colab import runtime
    runtime.unassign()